# Clase 26: Procesamiento de Texto — NLP Básico

**Idea central:** El procesamiento de texto convierte palabras en números para que los modelos puedan aprender a entender el lenguaje.

En esta clase aprenderemos a:
- Preprocesar texto en español
- Convertir texto en vectores numéricos con Bag of Words y TF-IDF
- Entrenar un clasificador de sentimientos
- Interpretar qué palabras son más importantes para cada clase

In [ ]:
# Celda 1: Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

print('Librerías cargadas correctamente')

In [ ]:
# Celda 2: Demostración con 5 frases simples (antes del dataset real)
# Primero entendemos el concepto con ejemplos pequeños

frases_demo = [
    'el producto es excelente muy recomendado',
    'muy malo no lo recomiendo para nada',
    'llegó rápido producto bien empacado',
    'producto defectuoso muy decepcionante',
    'calidad excelente precio justo llegó rápido'
]

# Bag of Words con CountVectorizer
cv = CountVectorizer()
X_demo = cv.fit_transform(frases_demo)

df_bow = pd.DataFrame(X_demo.toarray(), columns=cv.get_feature_names_out())
print('Bag of Words (5 frases de ejemplo):')
print(df_bow.to_string())
print(f'\nVocabulario: {len(cv.vocabulary_)} palabras')

In [ ]:
# Celda 3: TF-IDF con las mismas frases — observa las diferencias
tfidf_demo = TfidfVectorizer()
X_tfidf_demo = tfidf_demo.fit_transform(frases_demo)

df_tfidf_demo = pd.DataFrame(
    X_tfidf_demo.toarray(),
    columns=tfidf_demo.get_feature_names_out()
).round(3)

print('TF-IDF (mismas 5 frases):')
print(df_tfidf_demo.to_string())
print('\nObserva: palabras que aparecen en TODAS las frases (como "producto") tienen valores más bajos.')
print('Palabras únicas de cada frase tienen valores más altos.')

In [ ]:
# Celda 4: Cargar y explorar el dataset real de comentarios
df = pd.read_csv('../../datasets/comentarios_productos.csv')

print('Forma del dataset:', df.shape)
print('\nColumnas:', df.columns.tolist())
print('\nDistribución de sentimientos:')
print(df['sentimiento'].value_counts())
print('\nEjemplos de cada clase:')

for sentimiento in df['sentimiento'].unique():
    ejemplo = df[df['sentimiento'] == sentimiento]['comentario'].iloc[0]
    print(f'\n[{sentimiento}]: {ejemplo[:100]}...' if len(ejemplo) > 100 else f'\n[{sentimiento}]: {ejemplo}')

In [ ]:
# Celda 5: Función de preprocesamiento de texto
stop_words_es = {
    'el', 'la', 'los', 'las', 'un', 'una', 'de', 'del', 'al', 'en',
    'que', 'y', 'a', 'se', 'no', 'me', 'te', 'lo', 'le', 'su', 'por',
    'con', 'para', 'es', 'son', 'fue', 'era', 'pero', 'si', 'como',
    'mi', 'tu', 'ya', 'o', 'e', 'ni', 'hay', 'he', 'ha', 'han',
    'muy', 'mas', 'mas', 'tan', 'bien', 'mal', 'estar', 'ser'
}

def preprocesar_texto(texto):
    """Pipeline de limpieza de texto para NLP."""
    # 1. Convertir a minúsculas
    texto = texto.lower()
    # 2. Eliminar puntuación y caracteres especiales
    texto = re.sub(r'[^\w\s]', ' ', texto)
    # 3. Eliminar números
    texto = re.sub(r'\d+', '', texto)
    # 4. Eliminar stop words
    palabras = texto.split()
    palabras = [p for p in palabras if p not in stop_words_es and len(p) > 2]
    # 5. Unir y limpiar espacios extra
    return ' '.join(palabras)

# Probar la función
ejemplo = '¡El paquete llegó MUY TARDE!! Además, el producto estaba dañado. NO lo recomiendo para nada.'
print('Original:', ejemplo)
print('Procesado:', preprocesar_texto(ejemplo))

# Aplicar al dataset completo
df['texto_limpio'] = df['comentario'].apply(preprocesar_texto)
print('\nDataset con columna procesada:')
df[['comentario', 'texto_limpio']].head(3)

In [ ]:
# Celda 6: Vectorizar y entrenar el clasificador de sentimientos
X_raw = df['texto_limpio']
y = df['sentimiento']

# División estratificada (mantiene proporciones de clases)
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {len(X_train_raw)} comentarios')
print(f'Test: {len(X_test_raw)} comentarios')

# Vectorizar con TF-IDF
tfidf = TfidfVectorizer(max_features=500, ngram_range=(1, 2))
X_train = tfidf.fit_transform(X_train_raw)  # fit + transform en train
X_test = tfidf.transform(X_test_raw)         # SOLO transform en test (sin fit)

print(f'\nForma de X_train: {X_train.shape}  (comentarios x palabras/bigramas)')

# Entrenar clasificador
modelo = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
modelo.fit(X_train, y_train)

# Evaluar
y_pred = modelo.predict(X_test)
print('\n=== Resultados del clasificador de sentimientos ===')
print(classification_report(y_test, y_pred))

In [ ]:
# Celda 7: Matriz de confusión
cm = confusion_matrix(y_test, y_pred, labels=modelo.classes_)

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d',
            xticklabels=modelo.classes_,
            yticklabels=modelo.classes_,
            cmap='Blues')
plt.title('Matriz de confusión — Clasificador de sentimientos')
plt.ylabel('Sentimiento real')
plt.xlabel('Sentimiento predicho')
plt.tight_layout()
plt.show()
print('Las celdas de la diagonal principal son las predicciones correctas.')

In [ ]:
# Celda 8: Palabras más importantes para cada sentimiento
feature_names = tfidf.get_feature_names_out()
clases = modelo.classes_

fig, axes = plt.subplots(1, len(clases), figsize=(5 * len(clases), 5))

colores = ['#2ecc71', '#e74c3c', '#3498db', '#f39c12']

for idx, clase in enumerate(clases):
    coefs = modelo.coef_[idx]
    top_indices = coefs.argsort()[-12:][::-1]
    top_palabras = feature_names[top_indices]
    top_scores = coefs[top_indices]

    axes[idx].barh(top_palabras[::-1], top_scores[::-1],
                   color=colores[idx % len(colores)], edgecolor='white')
    axes[idx].set_title(f'Palabras clave:\n{clase}', fontsize=11)
    axes[idx].set_xlabel('Coeficiente')
    axes[idx].grid(axis='x', alpha=0.3)

plt.suptitle('Palabras más importantes por clase de sentimiento', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()
print('Coeficientes más altos = palabras más asociadas a esa clase de sentimiento.')

In [ ]:
# Celda 9: Función interactiva para predecir nuevos comentarios
def predecir_sentimiento(texto):
    """Predice el sentimiento de un comentario nuevo."""
    texto_limpio = preprocesar_texto(texto)
    X_nuevo = tfidf.transform([texto_limpio])
    prediccion = modelo.predict(X_nuevo)[0]
    probabilidades = modelo.predict_proba(X_nuevo)[0]
    
    print(f'Comentario:    "{texto}"')
    print(f'Texto limpio:   {texto_limpio}')
    print(f'Sentimiento:    {prediccion}')
    print('Probabilidades:')
    for clase, prob in sorted(zip(modelo.classes_, probabilidades), key=lambda x: -x[1]):
        barra = '█' * int(prob * 20)
        print(f'  {clase:12s}: {prob:.1%}  {barra}')
    print()

# Probar con diferentes comentarios
predecir_sentimiento('Llegó perfectamente y funciona de maravilla, muy satisfecho con la compra')
predecir_sentimiento('Una basura total, se rompió a los dos días y el servicio no responde')
predecir_sentimiento('El producto está bien, nada especial, cumple su función')

## Resumen del pipeline NLP

```
Texto crudo
   ↓ preprocesar_texto() — lower, puntuación, stop words
Texto limpio
   ↓ TfidfVectorizer.fit_transform() en train / .transform() en test
Matriz numérica (n_docs × n_features)
   ↓ LogisticRegression.fit()
Modelo entrenado
   ↓ .predict() / .predict_proba()
Sentimiento predicho
```

| Herramienta | Qué hace |
|---|---|
| `CountVectorizer` | Conteos de palabras (Bag of Words) |
| `TfidfVectorizer` | Pesos TF-IDF (penaliza palabras comunes) |
| `ngram_range=(1,2)` | Incluye bigramas como 'no recomiendo' |
| `LogisticRegression` | Clasificador lineal eficiente para texto |
| `coef_` | Importancia de cada palabra por clase |

**Recuerda:** Siempre usa `.fit_transform()` en train y `.transform()` en test para evitar data leakage.